# V5 v2 Residual Listwise Ranker

Train a second-stage ranker on cached candidates from the serving V4 configuration (`popularity_blend_weight=0.02`). Candidate retrieval runs once, then all epochs reuse the cache. The ranker starts as an identity ordering of V4 and is selected by validation multi-positive NDCG@10, not training loss.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

from pathlib import Path
import gc, hashlib, importlib.util, json, shutil, subprocess, sys, zipfile

DRIVE_ROOT = Path('/content/drive/MyDrive/recsys')
ADAPTIVE_ROOT = DRIVE_ROOT / 'data/gold/adaptive/v1'
BASE_GOLD_ROOT = DRIVE_ROOT / 'data/gold/two_tower/v1_colab'
WORK_ROOT = Path('/content/recsys_adaptive_eval')
SOURCE_ROOT = WORK_ROOT / 'runtime_v2'
UPLOAD_ROOT = WORK_ROOT / 'uploads'
V4_UPLOAD = UPLOAD_ROOT / 'v4'
V4_DIR = WORK_ROOT / 'artifacts/v4_sequence_retrieval'
V5_V2_DIR = WORK_ROOT / 'artifacts/v5_ranker_v2'
EVAL_DIR = WORK_ROOT / 'reports/v5_ranker_v2'
for path in [SOURCE_ROOT, V4_UPLOAD, V4_DIR, V5_V2_DIR, EVAL_DIR]:
    path.mkdir(parents=True, exist_ok=True)
print('V4 already restored:', (V4_DIR / 'best_model.pt').exists())

In [ ]:
required = {'pyarrow': 'pyarrow', 'pandas': 'pandas', 'torch': 'torch'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import torch
assert torch.cuda.is_available(), 'Select a Colab GPU runtime. L4 or A100 with High-RAM is recommended.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
required_paths = [
    ADAPTIVE_ROOT / 'sequence_examples/train',
    ADAPTIVE_ROOT / 'sequence_examples/validation',
    BASE_GOLD_ROOT / 'transforms/numeric_stats.json',
    BASE_GOLD_ROOT / 'vocabularies/user_active_degree',
    BASE_GOLD_ROOT / 'vocabularies/video_type',
    BASE_GOLD_ROOT / 'vocabularies/upload_type',
]
missing = [str(path) for path in required_paths if not path.exists()]
assert not missing, 'Missing inputs:\n' + '\n'.join(missing)
assert (ADAPTIVE_ROOT / 'sequence_examples/train/_SUCCESS').exists()
assert (ADAPTIVE_ROOT / 'sequence_examples/validation/_SUCCESS').exists()
print('Drive inputs ready')

## Refresh runtime code

Upload the latest `adaptive_eval_runtime.zip` from the project root.

In [ ]:
uploaded = files.upload()
runtime_names = [name for name in uploaded if Path(name).name.startswith('adaptive_eval_runtime') and name.endswith('.zip')]
assert runtime_names, 'Select adaptive_eval_runtime.zip'
runtime_zip = WORK_ROOT / 'adaptive_eval_runtime_v2.zip'
runtime_zip.write_bytes(uploaded[runtime_names[0]])
del uploaded
gc.collect()
if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir(parents=True)
with zipfile.ZipFile(runtime_zip) as archive:
    archive.extractall(SOURCE_ROOT)
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
assert (SOURCE_ROOT / 'scripts/train_adaptive_v5_ranker_v2.py').exists()
print('V5 v2 runtime ready')

## Restore V4 only if this is a new Colab runtime

If Notebook 18 ran in the same live runtime, the next upload cells skip automatically. Otherwise upload V4 manifest and parts in four batches.

In [ ]:
def upload_batch(destination: Path, label: str):
    if (V4_DIR / 'best_model.pt').exists():
        print('V4 checkpoint already restored; skipping', label)
        return
    print(label)
    uploaded = files.upload()
    for name, payload in uploaded.items():
        (destination / Path(name).name).write_bytes(payload)
    names = sorted(uploaded)
    del uploaded
    gc.collect()
    print('saved', len(names), 'files')

upload_batch(V4_UPLOAD, 'Select V4 manifest and parts 0000-0006')

In [ ]:
upload_batch(V4_UPLOAD, 'Select V4 parts 0007-0013')

In [ ]:
upload_batch(V4_UPLOAD, 'Select V4 parts 0014-0020')

In [ ]:
upload_batch(V4_UPLOAD, 'Select V4 parts 0021-0026')

In [ ]:
def sha256(path: Path, block_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while block := stream.read(block_size):
            digest.update(block)
    return digest.hexdigest()

def restore_v4():
    target = V4_DIR / 'best_model.pt'
    if target.exists():
        print('reusing:', target, target.stat().st_size)
        return
    manifests = sorted(V4_UPLOAD.glob('*manifest*.json'))
    assert manifests, 'V4 manifest is missing'
    manifest = json.loads(manifests[-1].read_text())
    for part in manifest['parts']:
        expected = V4_UPLOAD / part['filename']
        if not expected.exists():
            stem, number = part['filename'].split('.part_', 1)
            candidates = sorted(V4_UPLOAD.glob(f'{stem}*.part_{number}'))
            assert candidates, f'Missing {part["filename"]}'
            shutil.copy2(candidates[-1], expected)
    with target.open('wb') as output:
        for part in manifest['parts']:
            source = V4_UPLOAD / part['filename']
            assert source.stat().st_size == part['size_bytes']
            assert sha256(source) == part['sha256']
            with source.open('rb') as stream:
                shutil.copyfileobj(stream, output, length=8 * 1024 * 1024)
    assert sha256(target) == manifest['full_sha256']
    print('restored verified V4:', target, target.stat().st_size)

restore_v4()

## Train V5 v2

The expensive V4 item embeddings and candidates are generated once. Training then reuses those arrays for every epoch. The initial epoch-0 checkpoint preserves V4 ordering exactly.

In [ ]:
train_script = SOURCE_ROOT / 'scripts/train_adaptive_v5_ranker_v2.py'
command = [
    sys.executable, str(train_script),
    '--adaptive-root', str(ADAPTIVE_ROOT),
    '--base-gold-root', str(BASE_GOLD_ROOT),
    '--v4-dir', str(V4_DIR),
    '--output-dir', str(V5_V2_DIR),
    '--max-train-queries', '80000',
    '--max-validation-users', '0',
    '--train-candidates', '128',
    '--validation-candidates', '1000',
    '--popularity-blend-weight', '0.02',
    '--epochs', '12',
    '--batch-query-size', '64',
    '--retrieval-query-batch-size', '32',
    '--learning-rate', '0.0003',
    '--early-stopping-patience', '3',
    '--device', 'cuda',
]
subprocess.run(command, check=True)
metrics = json.loads((V5_V2_DIR / 'metrics.json').read_text())
print('accepted_over_v4:', metrics['accepted_over_v4'])
print('best_epoch:', metrics['best_epoch'])
print('best_validation_ndcg@10:', metrics['best_validation_ndcg@10'])

## Controlled full validation evaluation

This compares V5 v2 against the same V4 blend-0.02 Top-1000 candidates.

In [ ]:
evaluate_script = SOURCE_ROOT / 'scripts/evaluate_adaptive_v4_v5.py'
command = [
    sys.executable, str(evaluate_script),
    '--adaptive-root', str(ADAPTIVE_ROOT),
    '--base-gold-root', str(BASE_GOLD_ROOT),
    '--v4-dir', str(V4_DIR),
    '--v5-dir', str(V5_V2_DIR),
    '--output-dir', str(EVAL_DIR),
    '--split', 'validation',
    '--max-queries', '0',
    '--max-candidate-items', '0',
    '--retrieval-candidates', '1000',
    '--v5-candidate-blend-weight', '0.02',
    '--popularity-blend-grid', '0.0,0.02',
    '--query-batch-size', '32',
    '--device', 'cuda',
]
subprocess.run(command, check=True)
evaluation = json.loads((EVAL_DIR / 'adaptive_v4_v5_evaluation.json').read_text())
evaluation['ranker_delta_vs_v4_multi_positive']

## Download V5 v2 artifacts

The archive is small: checkpoint, config, metrics, history, and controlled evaluation only. Candidate caches are intentionally excluded.

In [ ]:
artifact_zip = Path('/content/v5_ranker_v2_artifacts.zip')
if artifact_zip.exists():
    artifact_zip.unlink()
with zipfile.ZipFile(artifact_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for root in [V5_V2_DIR, EVAL_DIR]:
        for path in root.rglob('*'):
            if path.is_file():
                archive.write(path, path.relative_to(WORK_ROOT))
print('artifact archive:', artifact_zip, artifact_zip.stat().st_size, 'bytes')
files.download(str(artifact_zip))